### Droughts, reelection, and insurance

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import geopandas as gpd

In [2]:
df_catnat = pd.read_csv('../data/gaspar/catnat_gaspar.csv', sep=';')

df_catnat['dat_deb'] = pd.to_datetime(df_catnat['dat_deb'], format='%Y-%m-%d')
df_catnat['dat_pub_jo'] = pd.to_datetime(df_catnat['dat_pub_jo'], format='%Y-%m-%d')
df_catnat['duree'] = (df_catnat['dat_pub_jo'] - df_catnat['dat_deb']).dt.days

df_catnat[['duree', 'lib_risque_jo']].groupby('lib_risque_jo').agg(['count', 'mean', 'std']).sort_values(('duree', 'count'), ascending=False).head(10)

duree              \
                                                     count        mean   
lib_risque_jo                                                            
Inondations et/ou Coulées de Boue                   145433   70.705369   
Sécheresse                                           46365  826.412984   
Mouvement de Terrain                                 32298   37.989287   
Tempête                                              16187   20.690678   
Chocs Mécaniques liés à l'action des Vagues           6773   25.462720   
Glissement de Terrain                                 3903  107.143992   
Poids de la Neige                                     2758  113.748731   
Mouvements de terrain différentiels consécutifs...    1543  588.787427   
Grêle                                                 1491   56.592220   
Inondations Remontée Nappe                            1430  319.219580   

                                                                
                                                           std  
lib_risque_jo                                                   
Inondations et/ou Coulées de Boue                    87.550285  
Sécheresse                                          715.050420  
Mouvement de Terrain                                130.691360  
Tempête                                              18.063089  
Chocs Mécaniques liés à l'action des Vagues          59.067270  
Glissement de Terrain                               138.837226  
Poids de la Neige                                    95.692776  
Mouvements de terrain différentiels consécutifs...  114.039780  
Grêle                                                20.345375  
Inondations Remontée Nappe                          248.694384

In [3]:
df_droughts = df_catnat[df_catnat['lib_risque_jo'] == 'Sécheresse']

In [4]:
df_spei = pd.read_csv('C:/Users/colin/Downloads/spei.csv', sep=',')
df_anomaly_temp = pd.read_csv('C:/Users/colin/Downloads/temperature_anomaly.csv', sep=',')
df_spei['lat'] = df_spei['ID'].str.split('_').str[0].astype(float)
df_spei['lon'] = df_spei['ID'].str.split('_').str[1].astype(float)
df_anomaly_temp['lat'] = df_anomaly_temp['ID'].str.split('_').str[0].astype(float)
df_anomaly_temp['lon'] = df_anomaly_temp['ID'].str.split('_').str[1].astype(float)

In [5]:
df_meta = pd.read_csv('C:/Users/colin/Downloads/coordonnees_grille_safran_lambert-2-etendu.csv', sep=';')
df_meta['LAT_DG'] = df_meta['LAT_DG'].str.replace(',', '.')
df_meta['LAT_DG'] = df_meta['LAT_DG'].astype(float)
df_meta['LON_DG'] = df_meta['LON_DG'].str.replace(',', '.')
df_meta['LON_DG'] = df_meta['LON_DG'].astype(float)
df_meta['LAMBX (hm)'] = df_meta['LAMBX (hm)'].astype(float)
df_meta['LAMBY (hm)'] = df_meta['LAMBY (hm)'].astype(float)
df_spei = df_spei.merge(df_meta, left_on=['lat', 'lon'], right_on=['LAMBX (hm)', 'LAMBY (hm)'], how='left')
df_anomaly_temp = df_anomaly_temp.merge(df_meta, left_on=['lat', 'lon'], right_on=['LAMBX (hm)', 'LAMBY (hm)'], how='left')
df_meta['ID'] = df_meta['LAMBX (hm)'].astype(int).astype(str) + '_' + df_meta['LAMBY (hm)'].astype(int).astype(str)

In [6]:
df_spei

,ID,year,month,time,def_pr,spei,lat,lon,LAMBX (hm),LAMBY (hm),LAT_DG,LON_DG
0,1000_23290,1970,1,1970-01-16 00:00:00,2.699792,1.142342,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
1,1000_23290,1970,2,1970-02-14 12:00:00,2.380476,1.007828,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
2,1000_23290,1970,3,1970-03-16 00:00:00,0.309462,0.076555,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
3,1000_23290,1970,4,1970-04-15 12:00:00,-0.294667,-0.238048,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
4,1000_23290,1970,5,1970-05-16 00:00:00,-1.796452,-1.019978,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
...,...,...,...,...,...,...,...,...,...,...,...,...
6568283,9960_25050,2024,12,2024-12-16 00:00:00,2.274086,1.019817,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402
6568284,9960_25050,2025,1,2025-01-16 00:00:00,1.936022,0.839519,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402
6568285,9960_25050,2025,2,2025-02-14 12:00:00,0.943690,0.310207,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402
6568286,9960_25050,2025,3,2025-03-16 00:00:00,-1.147043,-0.925408,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402


In [7]:
lau_2023 = gpd.read_file('../data/LAU_2023_EU/lau_2023_final.shp')
lau_2023 = lau_2023[lau_2023.CNTR_CODE == 'FR']
lau_2023 = lau_2023.to_crs(epsg=4326)

In [8]:
#find the closest ID in df_meta of each centroid of the city
lau_2023['x'] = lau_2023.geometry.centroid.x
lau_2023['y'] = lau_2023.geometry.centroid.y

def find_closest_id(row, df_meta):
    # Calculate the distance between the row's coordinates and all coordinates in df_meta
    distances = np.sqrt((df_meta['LAT_DG'] - row['y'])**2 + (df_meta['LON_DG'] - row['x'])**2)
    # Find the index of the closest point
    closest_index = distances.idxmin()
    # Return the ID of the closest point
    return df_meta.loc[closest_index, 'ID']

lau_2023['closest_id'] = lau_2023.apply(find_closest_id, axis=1, df_meta=df_meta)

C:\Users\colin\AppData\Local\Temp\ipykernel_50752\696722185.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  lau_2023['x'] = lau_2023.geometry.centroid.x
C:\Users\colin\AppData\Local\Temp\ipykernel_50752\696722185.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  lau_2023['y'] = lau_2023.geometry.centroid.y


In [9]:
df_droughts

,cod_nat_catnat,cod_commune,lib_commune,num_risque_jo,lib_risque_jo,dat_deb,dat_fin,dat_pub_arrete,dat_pub_jo,dat_maj,duree
5490,INTE0000771A,04001,Aiglun,59.0,Sécheresse,1998-01-01,1999-09-30,2000-12-27,2000-12-29,2022-05-24,1093
5491,INTE0000771A,04047,Champtercier,59.0,Sécheresse,1997-04-01,1998-06-30,2000-12-27,2000-12-29,2022-05-24,1368
5492,INTE0000771A,04128,Montfuron,59.0,Sécheresse,1997-04-01,1999-07-31,2000-12-27,2000-12-29,2022-05-24,1368
5493,INTE0000771A,09061,Les Bordes-sur-Arize,59.0,Sécheresse,1989-05-01,1990-12-31,2000-12-27,2000-12-29,2022-05-24,4260
5494,INTE0000771A,09307,Taurignan-Castet,59.0,Sécheresse,1989-05-01,1990-12-31,2000-12-27,2000-12-29,2022-05-24,4260
...,...,...,...,...,...,...,...,...,...,...,...
259848,INTE2430295A,31478,Saint-Félix-Lauragais,NaN,Sécheresse,2023-03-31,2023-06-29,2024-11-18,2024-12-02,2024-12-04,612
259855,INTE2430295A,11281,Pexiora,NaN,Sécheresse,2023-09-30,2023-12-30,2024-11-18,2024-12-02,2024-12-04,429
259858,INTE2430295A,66069,Espira-de-l'Agly,NaN,Sécheresse,2022-12-31,2023-12-30,2024-11-18,2024-12-02,2024-12-04,702
259860,INTE2430295A,26153,Laborel,NaN,Sécheresse,2023-03-31,2023-06-29,2024-11-18,2024-12-02,2024-12-04,612


In [17]:
df_spei

,ID,year,month,time,def_pr,spei,lat,lon,LAMBX (hm),LAMBY (hm),LAT_DG,LON_DG
0,1000_23290,1970,1,1970-01-16 00:00:00,2.699792,1.142342,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
1,1000_23290,1970,2,1970-02-14 12:00:00,2.380476,1.007828,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
2,1000_23290,1970,3,1970-03-16 00:00:00,0.309462,0.076555,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
3,1000_23290,1970,4,1970-04-15 12:00:00,-0.294667,-0.238048,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
4,1000_23290,1970,5,1970-05-16 00:00:00,-1.796452,-1.019978,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
...,...,...,...,...,...,...,...,...,...,...,...,...
6568283,9960_25050,2024,12,2024-12-16 00:00:00,2.274086,1.019817,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402
6568284,9960_25050,2025,1,2025-01-16 00:00:00,1.936022,0.839519,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402
6568285,9960_25050,2025,2,2025-02-14 12:00:00,0.943690,0.310207,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402
6568286,9960_25050,2025,3,2025-03-16 00:00:00,-1.147043,-0.925408,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402


In [ ]:
#for every row in df_droughts extract the mean SPEI from df_spei for the corresponding closest_id and date

def extract_spei(row, df_spei):
    year_start = row['dat_deb'].year -1
    month_start = row['dat_deb'].month
    year_end = row['dat_pub_jo'].year
    month_end = row['dat_pub_jo'].month

    #join based on lau_2023
    closest_id = lau_2023[lau_2023['LAU_ID']==row['cod_commune']]['closest_id'].values[0]
    df_spei_filtered = df_spei[df_spei['ID'] == closest_id]
    df_spei_filtered = df_spei_filtered[(df_spei_filtered['year'] >= year_start) & (df_spei_filtered['year'] <= year_end)]
    df_spei_filtered = df_spei_filtered[((df_spei_filtered['year'] != year_start)) | (df_spei_filtered['month'] < month_start)]
    df_spei_filtered = df_spei_filtered[((df_spei_filtered['year'] != year_end)) | (df_spei_filtered['month'] > month_end)]
    spei = df_spei_filtered['spei'].mean()
    return spei


In [ ]:
from tqdm import tqdm
df_spei

,ID,year,month,time,def_pr,spei,lat,lon,LAMBX (hm),LAMBY (hm),LAT_DG,LON_DG
0,1000_23290,1970,1,1970-01-16 00:00:00,2.699792,1.142342,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
1,1000_23290,1970,2,1970-02-14 12:00:00,2.380476,1.007828,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
2,1000_23290,1970,3,1970-03-16 00:00:00,0.309462,0.076555,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
3,1000_23290,1970,4,1970-04-15 12:00:00,-0.294667,-0.238048,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
4,1000_23290,1970,5,1970-05-16 00:00:00,-1.796452,-1.019978,1000.0,23290.0,1000.0,23290.0,47.7692,-4.34082
...,...,...,...,...,...,...,...,...,...,...,...,...
6568283,9960_25050,2024,12,2024-12-16 00:00:00,2.274086,1.019817,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402
6568284,9960_25050,2025,1,2025-01-16 00:00:00,1.936022,0.839519,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402
6568285,9960_25050,2025,2,2025-02-14 12:00:00,0.943690,0.310207,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402
6568286,9960_25050,2025,3,2025-03-16 00:00:00,-1.147043,-0.925408,9960.0,25050.0,9960.0,25050.0,49.4187,7.79402


In [24]:
from tqdm import tqdm

def extract_spei_optimized(df_droughts, df_spei, lau_2023):
    # Merge closest_id into df_droughts
    df_droughts = df_droughts.merge(lau_2023[['LAU_ID', 'closest_id']],
                                     left_on='cod_commune', right_on='LAU_ID',
                                     how='left')
    print(df_droughts.shape)
    # Preprocess SPEI
    #df_spei = preprocess_spei(df_spei)
    df_spei['time'] = pd.to_datetime(df_spei['time'], format='%Y-%m-%d %H:%M:%S')
    print(df_spei.shape)
    def get_spei(row):
        start_date = pd.to_datetime(row['dat_deb']) - pd.DateOffset(years=1)
        end_date = pd.to_datetime(row['dat_pub_jo'])
        mask = (
            (df_spei['ID'] == row['closest_id']) &
            (df_spei['time'] >= start_date) &
            (df_spei['time'] <= end_date)
        )
        return df_spei.loc[mask, 'spei'].mean()
    tqdm.pandas()
    df_droughts['mean_spei'] = df_droughts.apply(get_spei, axis=1)
    return df_droughts


In [26]:
from tqdm import tqdm
tqdm.pandas() 
extract_spei_optimized(df_droughts.iloc[0:1000], df_spei, lau_2023)

(1000, 13)
(6568288, 12)


,cod_nat_catnat,cod_commune,lib_commune,num_risque_jo,lib_risque_jo,dat_deb,dat_fin,dat_pub_arrete,dat_pub_jo,dat_maj,duree,LAU_ID,closest_id,mean_spei
0,INTE0000771A,04001,Aiglun,59.0,Sécheresse,1998-01-01,1999-09-30,2000-12-27,2000-12-29,2022-05-24,1093,04001,9080_19050,0.489216
1,INTE0000771A,04047,Champtercier,59.0,Sécheresse,1997-04-01,1998-06-30,2000-12-27,2000-12-29,2022-05-24,1368,04047,9080_19050,0.505105
2,INTE0000771A,04128,Montfuron,59.0,Sécheresse,1997-04-01,1999-07-31,2000-12-27,2000-12-29,2022-05-24,1368,04128,8680_18730,0.670207
3,INTE0000771A,09061,Les Bordes-sur-Arize,59.0,Sécheresse,1989-05-01,1990-12-31,2000-12-27,2000-12-29,2022-05-24,4260,09061,5240_17930,-0.080049
4,INTE0000771A,09307,Taurignan-Castet,59.0,Sécheresse,1989-05-01,1990-12-31,2000-12-27,2000-12-29,2022-05-24,4260,09307,5000_17850,-0.078838
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,INTE0100409A,82102,Mansonville,59.0,Sécheresse,1989-05-01,1990-09-01,2001-07-06,2001-07-18,2022-05-24,4461,82102,4840_18890,-0.093746
996,INTE0100409A,82102,Mansonville,59.0,Sécheresse,1989-05-01,1990-09-01,2001-07-06,2001-07-18,2022-05-24,4461,82102,4840_18890,-0.093746
997,INTE0100409A,82102,Mansonville,59.0,Sécheresse,1992-03-01,1992-06-30,2001-07-06,2001-07-18,2022-05-24,3426,82102,4840_18890,-0.033315
998,INTE0100409A,82102,Mansonville,59.0,Sécheresse,1992-03-01,1992-06-30,2001-07-06,2001-07-18,2022-05-24,3426,82102,4840_18890,-0.033315


In [15]:
import pandas as pd
from tqdm.notebook import tqdm
import pandas as pd

tqdm.pandas()  # 
df = pd.DataFrame({'a': range(100)})

df['b'] = df.progress_apply(lambda x: x['a'] ** 2, axis=1)

  0%|          | 0/100 [00:00<?, ?it/s]

In [62]:
#for every row in df_droughts extract the mean SPEI from df_spei for the corresponding closest_id and date
# Map closest_id from lau_2023 to df_droughts based on cod_commune
df_droughts = df_catnat[df_catnat['lib_risque_jo'] == 'Sécheresse']
df_droughts = df_droughts.merge(lau_2023[['LAU_ID', 'closest_id']], left_on='cod_commune', right_on='LAU_ID', how='left')

# Filter df_spei to include only relevant IDs and years
relevant_ids = df_droughts['closest_id'].unique()
# Use a generator to filter df_spei in chunks to avoid memory issues
def filter_spei_in_chunks(df_spei, relevant_ids, chunk_size=100000):
    for start in range(0, len(df_spei), chunk_size):
        chunk = df_spei.iloc[start:start + chunk_size]
        yield chunk[chunk['ID'].isin(relevant_ids)]

# Combine filtered chunks into a single DataFrame
df_spei_filtered = pd.concat(filter_spei_in_chunks(df_spei, relevant_ids), ignore_index=True)

# Add year and month ranges to df_droughts
df_droughts['year_start'] = df_droughts['dat_deb'].dt.year - 1
df_droughts['month_start'] = df_droughts['dat_deb'].dt.month
df_droughts['year_end'] = df_droughts['dat_pub_jo'].dt.year
df_droughts['month_end'] = df_droughts['dat_pub_jo'].dt.month

# Merge df_droughts with df_spei_filtered on closest_id and filter by date ranges
# Use a generator to process df_spei_filtered in chunks to avoid memory issues
def process_spei_in_chunks(df_spei_filtered, df_droughts, chunk_size=100000):
    for start in range(0, len(df_spei_filtered), chunk_size):
        chunk = df_spei_filtered.iloc[start:start + chunk_size]
        chunk = chunk.merge(df_droughts, left_on='ID', right_on='closest_id', how='inner')
        chunk = chunk[
            ((chunk['year'] > chunk['year_start']) | 
             ((chunk['year'] == chunk['year_start']) & (chunk['month'] >= chunk['month_start']))) &
            ((chunk['year'] < chunk['year_end']) | 
             ((chunk['year'] == chunk['year_end']) & (chunk['month'] <= chunk['month_end'])))
        ]
        yield chunk

# Combine processed chunks into a single DataFrame
df_spei_merged = pd.concat(process_spei_in_chunks(df_spei_filtered, df_droughts), ignore_index=True)

# Calculate mean SPEI for each drought event
mean_spei = df_spei_merged.groupby('cod_nat_catnat')['spei'].mean().reset_index()
mean_spei.rename(columns={'spei': 'mean_spei'}, inplace=True)

# Merge the mean SPEI back into df_droughts
df_droughts = df_droughts.merge(mean_spei, on='cod_nat_catnat', how='left')


MemoryError: Unable to allocate 105. MiB for an array with shape (4, 3443504) and data type float64

In [58]:
df_spei.spei.mean()

0.13449949181732762

In [56]:
#test with one row
from tqdm import tqdm
for i, row in tqdm(df_droughts.iterrows(), total=df_droughts.shape[0]):
    spei = extract_spei(row, df_spei)
    print(spei)
    

  0%|          | 1/46365 [00:00<5:50:46,  2.20it/s]

0.4393493468625635


  0%|          | 2/46365 [00:00<5:16:13,  2.44it/s]

0.48217408273082757


  0%|          | 3/46365 [00:01<5:06:27,  2.52it/s]

...

0.025526620710838385


  0%|          | 59/46365 [00:22<5:06:33,  2.52it/s]

0.025526620710838385


  0%|          | 59/46365 [00:23<5:02:09,  2.55it/s]


KeyboardInterrupt: 